# Convert Retrieval Results to Format A

This notebook transforms `data_local_retrieval_results.json` into **Format A** (separate chunks.json) for chunk-level assertion evaluation.

Format A structure:
```json
{
  "question_id": "...",
  "question_text": "...",
  "chunks": [
    {"text": "...", "rank": 0, "chunk_id": 0},
    ...
  ]
}
```

## Section 1: Configure Notebook Paths and Dependencies

In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
import json
from pathlib import Path
from typing import Any, TypedDict

# Define paths (notebook is in chunk_format_conversion/, example_answers is sibling directory)
notebook_dir = Path()
input_file = (
    notebook_dir
    / ".."
    / "example_answers"
    / "vector_rag_long_context"
    / "data_global_retrieval_results.json"
)
output_file = notebook_dir / "chunks.json"

print(f"Input file: {input_file.resolve()}")
print(f"Output file: {output_file.resolve()}")
print(f"Input exists: {input_file.exists()}")

## Section 2: Load `data_local_retrieval_results.json`

In [ ]:
try:
    with input_file.open() as f:
        source_data = json.load(f)

    print(f"✓ Successfully loaded {len(source_data)} records from {input_file.name}")
    print(f"  Type: {type(source_data)}")

except FileNotFoundError:
    print(f"✗ File not found: {input_file}")
    source_data = []

except json.JSONDecodeError as e:
    print(f"✗ JSON parsing error: {e}")
    source_data = []

## Section 3: Profile Source JSON Structure

In [ ]:
if source_data:
    # Inspect first record
    first_record = source_data[0]
    print("=== FIRST RECORD STRUCTURE ===")
    print(f"Top-level keys: {list(first_record.keys())}")
    print(f"\nquestion_id: {first_record.get('question_id')}")
    print(f"text: {first_record.get('text')[:80]}...")

    print("\ncontext[0]:")
    if first_record.get("context"):
        context_item = first_record["context"][0]
        print(f"  Keys: {list(context_item.keys())}")
        print(f"  chunk_id: {context_item.get('chunk_id')}")
        print(f"  text length: {len(context_item.get('text', ''))}")

    print(f"\nTotal records: {len(source_data)}")
    print(f"Context items in first record: {len(first_record.get('context', []))}")

## Section 4: Define Option A Target Schema

**Option A Format:**
- `question_id` (string): Unique question identifier
- `question_text` (string): The question text
- `chunks` (array): Retrieved chunks with `text`, `rank`, and `chunk_id`

**Field Mapping:**
| Source | Target | Notes |
|--------|--------|-------|
| `question_id` | `question_id` | Direct copy |
| `text` | `question_text` | Rename "text" → "question_text" |
| `context[*]` | `chunks[]` | Convert array to chunks |
| `context[i].text` | `chunks[i].text` | Direct copy |
| `context[i].chunk_id` | `chunks[i].chunk_id` | Rename "chunk_id" |
| (array index) | `chunks[i].rank` | Use 0-based index as rank |

In [ ]:
# Define Option A schema template
class ChunkItem(TypedDict):
    """A single retrieved chunk in Option A format."""

    text: str
    rank: int
    chunk_id: int | str


class OptionARecord(TypedDict):
    """A per-question record in Option A format."""

    question_id: str
    question_text: str
    chunks: list[ChunkItem]


# Schema validation template
REQUIRED_FIELDS = {
    "question_id": str,
    "question_text": str,
    "chunks": list,
}

CHUNK_REQUIRED_FIELDS = {
    "text": str,
    "rank": int,
    "chunk_id": (int, str),
}

print("✓ Option A schema defined")
print(f"  Required top-level fields: {list(REQUIRED_FIELDS.keys())}")
print(f"  Required chunk fields: {list(CHUNK_REQUIRED_FIELDS.keys())}")

## Section 5: Implement Record Transformation to Option A

In [ ]:
def transform_record(source_record: dict[str, Any]) -> dict[str, Any]:
    """Transform source record to Option A format.

    - question_id: copied as-is
    - question_text: renamed from 'text'
    - chunks: array with rank (0-based index) and chunk_id preserved
    """
    transformed = {
        "question_id": source_record.get("question_id"),
        "question_text": source_record.get("text"),
        "chunks": [],
    }

    # Transform context array to chunks with rank
    context_items = source_record.get("context", [])
    for rank, context_item in enumerate(context_items):
        chunk = {
            "text": context_item.get("text"),
            "rank": rank,  # 0-based index
            "chunk_id": context_item.get("chunk_id"),
        }
        transformed["chunks"].append(chunk)

    return transformed


# Test transformation on first record
if source_data:
    test_transformed = transform_record(source_data[0])
    print("✓ Transformation function created and tested")
    print("\nFirst record after transformation:")
    print(f"  question_id: {test_transformed['question_id']}")
    print(f"  question_text: {test_transformed['question_text'][:60]}...")
    print(f"  chunks: {len(test_transformed['chunks'])} items")
    print("\n  First chunk:")
    if test_transformed["chunks"]:
        first_chunk = test_transformed["chunks"][0]
        print(f"    text: {first_chunk['text'][:60]}...")
        print(f"    rank: {first_chunk['rank']}")
        print(f"    chunk_id: {first_chunk['chunk_id']}")

## Section 6: Validate Transformed Output

In [ ]:
def validate_record(record: dict[str, Any], _record_idx: int) -> tuple[bool, list[str]]:
    """Validate a transformed record against Option A schema.

    Returns: (is_valid, list of error messages)
    """
    errors = []

    # Check top-level required fields
    if not record.get("question_id"):
        errors.append("Missing or empty question_id")
    if not record.get("question_text"):
        errors.append("Missing or empty question_text")
    if "chunks" not in record:
        errors.append("Missing chunks array")

    # Check chunks array
    chunks = record.get("chunks", [])
    if not isinstance(chunks, list):
        errors.append("chunks is not a list")
    else:
        for chunk_idx, chunk in enumerate(chunks):
            if not chunk.get("text"):
                errors.append(f"chunk[{chunk_idx}] missing text")
            if not isinstance(chunk.get("rank"), int):
                errors.append(f"chunk[{chunk_idx}] missing or non-integer rank")
            if chunk.get("chunk_id") is None:
                errors.append(f"chunk[{chunk_idx}] missing chunk_id")

    return len(errors) == 0, errors


# Transform all records
transformed_data = []
invalid_records = []

if source_data:
    for idx, source_record in enumerate(source_data):
        transformed = transform_record(source_record)
        is_valid, errors = validate_record(transformed, idx)

        if is_valid:
            transformed_data.append(transformed)
        else:
            invalid_records.append({"record_idx": idx, "errors": errors})

    print("✓ Transformation complete")
    print(f"  Total source records: {len(source_data)}")
    print(f"  Valid transformed records: {len(transformed_data)}")
    print(f"  Invalid records: {len(invalid_records)}")

    if invalid_records:
        print("\n⚠ Invalid records:")
        for invalid in invalid_records[:3]:  # Show first 3
            print(f"  Record {invalid['record_idx']}: {invalid['errors']}")

## Section 7: Write Option A JSON File

In [ ]:
if transformed_data:
    try:
        with output_file.open("w") as f:
            json.dump(transformed_data, f, indent=2)

        print(
            f"✓ Successfully wrote {len(transformed_data)} records to {output_file.name}"
        )
        print(f"  File size: {output_file.stat().st_size:,} bytes")

    except OSError as e:
        print(f"✗ Error writing output file: {e}")

## Section 8: Run Sanity Checks on Saved Output

In [ ]:
if output_file.exists():
    # Reload written file
    with output_file.open() as f:
        verified_data = json.load(f)

    print("=== SANITY CHECKS ===\n")

    # Check record count
    print("Record counts:")
    print(f"  Source: {len(source_data)}")
    print(f"  Transformed: {len(transformed_data)}")
    print(f"  Verified (reloaded): {len(verified_data)}")

    if len(verified_data) == len(source_data):
        print("  ✓ All records preserved")
    else:
        print("  ✗ Record count mismatch!")

    # Sample first record
    print("\nFirst record sample:")
    first = verified_data[0]
    print(f"  question_id: {first['question_id']}")
    print(f"  question_text: {first['question_text'][:70]}...")
    print(f"  chunks: {len(first['chunks'])} items")
    print("\n  First 3 chunks:")
    for i, chunk in enumerate(first["chunks"][:3]):
        print(
            f"    [{i}] rank={chunk['rank']}, chunk_id={chunk['chunk_id']}, text_len={len(chunk['text'])}"
        )

    # Verify rank ordering
    print("\nRank verification (first record):")
    first_ranks = [c["rank"] for c in first["chunks"]]
    expected_ranks = list(range(len(first["chunks"])))
    if first_ranks == expected_ranks:
        print(f"  ✓ Ranks are sequential from 0: {first_ranks[:5]}...")
    else:
        print("  ✗ Ranks are not sequential!")

    print("\n✓ Conversion complete and verified!")
else:
    print("Output file not found!")